In [1]:
# import sys
# !{sys.executable} -m ensurepip --upgrade


In [2]:
# !{sys.executable} -m pip install pyfonts

In [3]:
import gcsfs
import pandas as pd
import os
from _utils import GCS_FILE_PATH
import geopandas as gpd
import folium
import altair as alt

import matplotlib.pyplot as plt
from pywaffle import Waffle
from pypalettes import load_palette
from pyfonts import load_google_font
from process_ntd import *
from chart_ideas import *

In [4]:
df = pd.read_parquet(
    f"{GCS_FILE_PATH}expenditure_funding.parquet",
    filesystem = gcsfs.GCSFileSystem()
)

In [5]:
df_CA = subset_california(df)

In [6]:
df_CA.head(2)

,key,ntd_id,year,legacy_ntd_id_funding,agency_status_funding,census_year_funding,last_report_year_funding,reporter_type_funding,reporting_module_funding,uace_code_funding,...,reporter_type,reporting_module,uace_code,uza_area_sq_miles,primary_uza_name_opex,uza_population,source_agency,source_city,source_state,cpi
0,5793309ba1d86c4eabd47bd8c02a4480,91063,2021,9R02-007,Inactive,2010,2018,Rural Reporter,Rural,6,...,Rural Reporter,Rural,-9999,0.0,N/A,0,Calaveras County Department of Public Works,San Andreas,CA,270.969750
1,ff895cf4777f08302911f958c8a2be1f,91063,2017,9R02-007,Inactive,2010,2018,Rural Reporter,Rural,6,...,Rural Reporter,Rural,-9999,0.0,N/A,0,Calaveras County Department of Public Works,San Andreas,CA,245.119583


## Example Charts 

In [7]:
df_CA.columns

Index(['key', 'ntd_id', 'year', 'legacy_ntd_id_funding',
       'agency_status_funding', 'census_year_funding',
       'last_report_year_funding', 'reporter_type_funding',
       'reporting_module_funding', 'uace_code_funding',
       'uza_area_sq_miles_funding', 'primary_uza_name',
       'uza_population_funding', '_2024_status', 'source_agency_funding',
       'source_city_funding', 'source_state_funding', 'operating_total',
       'operating_federal', 'operating_state', 'operating_local',
       'operating_other', 'capital_total', 'capital_federal', 'capital_state',
       'capital_local', 'capital_other', 'total_capital_expenditures',
       'rolling_stock_expenditures', 'facilities_expenditures',
       'other_expenditures', 'legacy_ntd_id_capex', 'agency_status_capex',
       'census_year_capex', 'last_report_year_capex', 'reporter_type_capex',
       'reporting_module_capex', 'uace_code_capex', 'uza_area_sq_miles_capex',
       'uza_name', 'uza_population_capex', 'source_agency_

### 1. Operating Funding vs. Expenses over time

In [8]:
operating_trend = (
    df_CA.groupby("year")[["operating_total", "operating_expenses_total"]]
    .sum()
    .reset_index()
    .melt("year", var_name="metric", value_name="amount")
    .assign(metric_type=lambda x: x["metric"].map({
        "operating_total": "Funding",
        "operating_expenses_total": "Expenses"
    }))
)

chart = (
    alt.Chart(operating_trend)
    .mark_line(point=True)
    .encode(
        x=alt.X("year:O", title="Year"),
        y=alt.Y("amount:Q", title="Amount ($)", axis=alt.Axis(format="~s")),
        color=alt.Color(
            "metric_type:N", title=None,
            scale=alt.Scale(
                domain=["Funding", "Expenses"],
                range=["#2C7FB8", "#D95F59"]
            )
        ),
        tooltip=[
            alt.Tooltip("year:O", title="Year"),
            alt.Tooltip("metric_type:N", title=""),
            alt.Tooltip("amount:Q", title="Amount", format=",.0f")
        ]
    )
    .properties(width=400, height=100, title="Operating Funding vs. Operating Expenses")
)

chart

alt.Chart(...)

In [9]:
capital_gap = (
    df_CA.groupby("year")[["capital_total", "total_capital_expenditures"]]
    .sum()
    .reset_index()
    .assign(gap=lambda x: x["capital_total"] - x["total_capital_expenditures"])
)

chart = (
    alt.Chart(capital_gap)
    .mark_bar()
    .encode(
        x=alt.X("year:O", title="Year"),
        y=alt.Y("gap:Q", title="Capital Funding − Expenditures ($)", axis=alt.Axis(format="~s")),
        color=alt.condition(
            alt.datum.gap >= 0, alt.value("#2C7FB8"), alt.value("#D95F59")
        ),
        tooltip=[
            alt.Tooltip("year:O", title="Year"),
            alt.Tooltip("capital_total:Q", title="Capital Funding", format=",.0f"),
            alt.Tooltip("total_capital_expenditures:Q", title="Capital Expenditures", format=",.0f"),
            alt.Tooltip("gap:Q", title="Gap", format=",.0f"),
        ],
    )
    .properties(width=500, height=200, title="Capital Funding Gap")
)

chart

alt.Chart(...)

In [10]:
chart = percent_stacked_bar(
    df_CA,
    group_column="year",
    category_columns={
        "Federal": "operating_federal",
        "State": "operating_state",
        "Local": "operating_local",
        "Other": "operating_other"
    },
    chart_title="Operating Funding Composition",
    y_title="Share of Operating Funding (%)",
    colors=["#2C7FB8", "#7FCDBB", "#FEC44F", "#D95F59"]
)

chart


alt.Chart(...)

Agency-level funding vs. expenses scatter

In [16]:
from sklearn.metrics import r2_score

expense_cols = {
    "Vehicle Operations": "operating_expenses_vehicle_operations",
    "Vehicle Maintenance": "operating_expenses_vehicle_maintenance",
    "Nonvehicle Maintenance": "operating_expenses_nonvehicle_maintenance",
    "General Administration": "operating_expenses_general_administration"
}

df_2024 = df[df["year"] == 2021]
charts = []

for name, col in expense_cols.items():
    d = df_2024[["operating_total", col]].dropna()
    d = d[(d["operating_total"] > 0) & (d[col] > 0)]

    r2 = r2_score(
        d[col],
        __import__("numpy").polyval(
            __import__("numpy").polyfit(d["operating_total"], d[col], 1),
            d["operating_total"]
        )
    )

    charts.append(
        scatter_plot_regression(
            df_2024,
            "operating_total",
            col,
            tooltip_columns=["ntd_id", "operating_total", col],
            chart_title=f"Funding vs. {name} — 2024 (R² = {r2:.2f})"
        ).properties(width=300, height=200)
    )

alt.vconcat(
    alt.hconcat(charts[0], charts[1]),
    alt.hconcat(charts[2], charts[3])
)


alt.VConcatChart(...)

In [19]:
capital_cols = {
    "Rolling Stock": "rolling_stock_expenditures",
    "Facilities": "facilities_expenditures",
    "Other": "other_expenditures"
}

df_2021 = df[df["year"] == 2024]
charts = []

for name, col in capital_cols.items():
    d = df_2021[["capital_total", col]].dropna()
    d = d[(d["capital_total"] > 0) & (d[col] > 0)]

    r2 = r2_score(
        d[col],
        np.polyval(np.polyfit(d["capital_total"], d[col], 1), d["capital_total"])
    )

    charts.append(
        scatter_plot_regression(
            df_2021,
            "capital_total",
            col,
            tooltip_columns=["ntd_id", "capital_total", col],
            chart_title=f"Capital Funding vs. {name} — 2021 (R² = {r2:.2f})"
        ).properties(width=300, height=220)
    )

alt.vconcat(
    alt.hconcat(charts[0], charts[1]),
    charts[2]
)

alt.VConcatChart(...)

Across NTD agencies, agencies with more capital funding tend to spend more on facilities, while spending on rolling stock and other projects varies more between agencies.

Cost Per Capita by Urbanized Area

This chart shows how **operating cost per capita varies with urbanized area (UZA) population in 2024**. Each circle represents a UZA, with population on the x-axis and operating cost per person on the y-axis. The logarithmic population scale makes it easier to compare both small and very large UZAs. Circle size represents total operating expenses, helping highlight the largest agencies. Overall, the chart can reveal whether larger urbanized areas tend to have lower or higher operating costs per resident and identify unusually high-cost or low-cost UZAs.

In [8]:
per_capita = (
    df_CA[df_CA["year"] == 2024]
    .groupby("primary_uza_name")
    .agg(opex=("operating_expenses_total", "sum"),
         pop=("uza_population", "max"))
    .reset_index()
    .assign(cost_per_capita=lambda x: x["opex"] / x["pop"])
    .query("pop > 0")
)

chart = (
    alt.Chart(per_capita)
    .mark_circle(opacity=0.7)
    .encode(
        x=alt.X("pop:Q", title="UZA Population", scale=alt.Scale(type="log")),
        y=alt.Y("cost_per_capita:Q", title="Operating Cost per Capita ($)"),
        size=alt.Size("opex:Q", title="Total Opex", legend=None),
        color=alt.value("#2C7FB8"),
        tooltip=["primary_uza_name:N", alt.Tooltip("pop:Q", format=","),
                 alt.Tooltip("cost_per_capita:Q", format="$.2f")]
    )
    .properties(width=500, height=300, title="Operating Cost per Capita vs. Urbanized Area Size — 2024")
)
chart

alt.Chart(...)

funding mix trend by reporter type

This chart shows how **transit funding sources have changed over time across different reporter types**. Each panel represents a reporter type, while the stacked areas show the percentage of operating funding coming from federal, state, local, and other sources. Because the chart is normalized to 100%, it focuses on the **composition of funding rather than the total dollar amount**, making it easy to compare how funding reliance differs across reporter types and changes over the years.

In [9]:
mix_by_type = (
    df_CA.groupby(["year", "reporter_type"])[["operating_federal","operating_state","operating_local","operating_other"]]
    .sum()
    .reset_index()
    .melt(["year","reporter_type"], var_name="source", value_name="amount")
)

chart = (pop = df_CA["uza_population"].fillna(0)

df_CA["size_bucket"] = pd.qcut(pop.rank(method="first"), 4,
                                labels=["Small","Mid-Small","Mid-Large","Large"])

df_CA["admin_share"] = df_CA["operating_expenses_general_administration"] / df_CA["operating_expenses_total"]

heat = df_CA.groupby(["year","size_bucket"], observed=True)["admin_share"].mean().reset_index()

chart = (
    alt.Chart(heat)
    .mark_rect()
    .encode(
        x=alt.X("year:O", title="Year"),
        y=alt.Y("size_bucket:N", title="Agency Size", sort=["Small","Mid-Small","Mid-Large","Large"]),
        color=alt.Color("admin_share:Q", title="Admin Share", scale=alt.Scale(scheme="orangered"), axis=alt.Axis(format="%")),
        tooltip=["year:O","size_bucket:N", alt.Tooltip("admin_share:Q", format=".1%")]
    )
    .properties(width=450, height=180, title="General Administration Overhead by Agency Size Over Time")
)
chart

    alt.Chart(mix_by_type)
    .transform_joinaggregate(total="sum(amount)", groupby=["year","reporter_type"])
    .transform_calculate(pct="datum.amount / datum.total")
    .mark_area()
    .encode(
        x=alt.X("year:O", title=None),
        y=alt.Y("pct:Q", stack="normalize", axis=alt.Axis(format="%"), title="Share"),
        color=alt.Color("source:N", scale=alt.Scale(scheme="tableau10")),
        facet=alt.Facet("reporter_type:N", columns=3, title=None),
        tooltip=["year:O","source:N", alt.Tooltip("pct:Q", format=".1%")]
    )
    .resolve_scale(y="shared")
    .properties(width=150, height=150, title="Funding Source Mix by Reporter Type")
)
chart

alt.Chart(...)